# AI Product Mini-Projekt 

- Laden Finanz-Daten für beliebige Firmen über SEC-API 
- Berechne Bewertungsmultiples
- Stelle sie dar über eine Dash-GUI 
- Ermögliche das Stellen von Fragen über LLM
- Binde bei Beantwortung 10-K Filings über RAG ein


## Setup

## Architektur des Mini-Projekts

Dieses Notebook ist bewusst in drei Ebenen getrennt:

1. `main_api` liefert generische SEC-API-Funktionen und kann auch in anderen Projekten wiederverwendet werden.
2. `teaching_dash_minimal_utils.py` enthaelt notebook-spezifische Hilfsfunktionen wie Filing-Download, Text-Extraktion und Kennzahlen-Loading.
3. Das Notebook selbst zeigt nur die Orchestrierung: Konfiguration, Beispielabfragen, RAG-Logik und Dash-Oberflaeche.

Dadurch bleibt das Notebook lesbar, waehrend die eigentliche Implementierung in Python-Dateien gepflegt wird.

In [ ]:
# Falls noetig, einmalig ausfuehren:
# %pip install -r requirements.txt

import os
import pandas as pd
# Library zum Erstellen von Grafiken
import plotly.express as px
import plotly.graph_objects as go
# Library zum Erstellen von Web-Dashboards
from dash import Dash, dcc, html, Input, Output, State
# Library zum Abrufen von Finanzdaten
import requests
from getpass import getpass

try:
    import teaching_dash_minimal_utils
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "teaching_dash_minimal_utils.py oder main_api.py wurde nicht gefunden. "
        "Bitte Notebook, teaching_dash_minimal_utils.py und main_api.py im selben Ordner bereitstellen "
        "oder das Projekt ueber start_notebook_env.sh starten."
    ) from exc

Die Utilities wurden in die Datei `teaching_dash_minimal_utils.py` ausgelagert.

Die folgende Zelle importiert diese Hilfsfunktionen fuer das Notebook. Bitte fuehren Sie sie aus, auch wenn wir ihren Inhalt nicht weiter besprechen.

Wichtige Beispiele aus dem Modul sind:

```python
load_three_metrics(cik: int, start_year: int, end_year: int) -> pd.DataFrame
FilingDownloader(...)
extract_clean_10k_text(filing_paths: dict, min_chars: int) -> tuple[str, str]
```

In [ ]:
%%capture

from teaching_dash_minimal_utils import (
    SEC_HEADERS,
    FilingDownloader,
    _build_openrouter_url,
    extract_clean_10k_text,
    load_three_metrics,
)

In [ ]:
# Konfiguration
BIG_TECH = {
    "Apple": 320193,
    "Microsoft": 789019,
    "Alphabet": 1652044,
    "Amazon": 1018724,
    "Meta": 1326801,
    "NVIDIA": 1045810,
    "Tesla": 1318605,
}


# Email fuer EDGAR API, damit die Anfragen nicht blockiert werden
SEC_USER_AGENT = "EdinDzeko@Goal.com"
SEC_HEADERS["User-Agent"] = SEC_USER_AGENT


metrics_df = load_three_metrics(BIG_TECH["Meta"], start_year=2001, end_year=2024)
metrics_df

,year,metric,value,form,mode
0,2001,diluted_earnings,NaN,10-K,direct
1,2001,capex,NaN,10-K,normalized_abs
2,2001,fcf,NaN,10-K,direct_from_companyfacts
3,2002,diluted_earnings,NaN,10-K,direct
4,2002,capex,NaN,10-K,normalized_abs
...,...,...,...,...,...
67,2023,capex,2.726600e+10,10-K,normalized_abs
68,2023,fcf,4.384700e+10,10-K,direct_from_companyfacts
69,2024,diluted_earnings,2.386000e+01,10-K,direct
70,2024,capex,3.725600e+10,10-K,normalized_abs


## LLM-Funktion 

Default: Cloud-Free-Tier ueber OpenRouter. Erstellen Sie einen kostenlosen API-Key dafür im Netz. Alternativ: Eigener Open-AI-Key für leistungsfährigere Modelle.

In [22]:
import os
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_api_key:
    print("Bitte setzen Sie die Variable OPENROUTER_API_KEY mit Ihrem OpenRouter API-Schlüssel.")
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API Key eingeben (wird nicht angezeigt): ")


In [23]:


def ask_llm(question: str, context: str = "") -> str:
    """Cloud-Free-Tier ueber OpenRouter."""
    api_key = os.getenv("OPENROUTER_API_KEY", "")
    if not api_key:
        return "Kein OPENROUTER_API_KEY gesetzt."

    url = _build_openrouter_url()
    model = os.getenv("OPENROUTER_MODEL", "openrouter/auto")

    prompt = f"Kontext:\n{context}\n\nFrage: {question}\nAntworte kurz und klar."

    try:
        r = requests.post(
            url,
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
                # Empfohlen von OpenRouter, hilft bei einigen Setups.
                "HTTP-Referer": os.getenv("OPENROUTER_SITE_URL", "http://localhost"),
                "X-Title": os.getenv("OPENROUTER_APP_NAME", "SEC Teaching Notebook"),
            },
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0,
            },
            timeout=120,
        )

        if r.status_code >= 400:
            # Detailierte Antwort hilft bei 404/401-Diagnose direkt im Notebook.
            body = r.text[:500]
            if r.status_code == 404:
                return (
                    f"LLM-API Fehler 404: Endpoint oder Modell nicht gefunden. "
                    f"URL={url}, model={model}, body={body}"
                )
            return f"LLM-API Fehler {r.status_code}: {body}"

        data = r.json()
        return data.get("choices", [{}])[0].get("message", {}).get("content", "Keine Antwort erhalten.")
    except Exception as e:
        return f"LLM-API Fehler: {e}"

## RAG für letzte 2x 10-K Filings

Dieser Abschnitt baut den Retrieval-Kontext aus den letzten zwei verfuegbaren 10-K Filings direkt ueber die SEC-API auf. 

1. Frage wird in Tokens zerlegt (q_tokens), z. B. Wörter wie „cash“, „flow“, „meta“.
2. Die letzten 2 passenden 10-K Filings werden über SEC geholt.
3. Aus jedem Filing wird der Text extrahiert.
4. Der Text wird in kleinere Abschnitte (Chunks) geteilt.
5. Jeder Chunk wird ebenfalls in Tokens zerlegt (c_tokens).
6. Pro Chunk wird ein einfacher Relevanz-Score berechnet:
```python
len(q_tokens.intersection(c_tokens))
```
Es wird gezählt, wie viele verschiedene Wörter in Frage und Chunk gleichzeitig vorkommen. Das ist eine sehr einfache Relevanz-Quantifizierung. Alternativen würden z.B. die Distanz zwischen dem Durchschnitt aus Embedding-Vektoren über alle Token in Query und einzelnen Chunks berechnen.

7. Die Top-k Chunks mit dem höchsten Score werden ausgewählt.
Diese Chunks werden als RAG-Kontext an das LLM gegeben.





In [27]:
import re
import requests
from bs4 import BeautifulSoup



def _normalize_text(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", (s or "").lower())


def _tokenize_for_rag(text: str) -> set[str]:
    tokens = re.findall(r"[a-zA-Z0-9]{3,}", (text or "").lower())
    stopwords = {
        "the", "and", "for", "with", "from", "that", "this", "are", "was", "were", "have", "has",
        "ein", "eine", "und", "der", "die", "das", "ist", "mit", "von", "auf", "wie", "sind",
        "what", "when", "where", "which", "uber", "into", "also", "their", "its", "our",
    }
    return {t for t in tokens if t not in stopwords}


def _clean_whitespace(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def _html_to_text(html: str) -> str:
    soup = BeautifulSoup(html or "", "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    return _clean_whitespace(soup.get_text(" "))


def _extract_10k_text_from_submission_raw(raw_submission: str) -> str:
    docs = (raw_submission or "").split("<DOCUMENT>")
    best_text = ""

    for d in docs[1:]:
        m_type = re.search(r"<TYPE>\s*([^\s<]+)", d)
        doc_type = (m_type.group(1).strip().upper() if m_type else "")

        if doc_type in {"10-K", "10K"}:
            m_text = re.search(r"<TEXT>(.*)</TEXT>", d, flags=re.DOTALL | re.IGNORECASE)
            if not m_text:
                continue
            content = m_text.group(1)

            if "<html" in content.lower() or "<table" in content.lower():
                cand = _html_to_text(content)
            else:
                cand = _clean_whitespace(content)

            if len(cand) > len(best_text):
                best_text = cand

    return best_text


def _fetch_complete_submission_text(cik: int, accession_number: str, headers: dict) -> str:
    accession_no_nodashes = accession_number.replace("-", "")
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_nodashes}/{accession_number}.txt"
    r = requests.get(url, headers=headers, timeout=120)
    r.raise_for_status()
    return r.text


def _find_latest_two_10k_filings_api(company_name: str) -> list[dict]:
    cik = BIG_TECH[company_name]
    downloader = FilingDownloader(download_dir="./downloads", user_agent=SEC_USER_AGENT)

    submissions = downloader.get_all_submissions(cik)
    recent = submissions.get("filings", {}).get("recent", {})

    forms = recent.get("form", [])
    filing_dates = recent.get("filingDate", [])
    accession_numbers = recent.get("accessionNumber", [])
    fiscal_years = recent.get("fiscalYear", [])
    report_dates = recent.get("reportDate", [])

    candidates = []
    for i, form in enumerate(forms):
        if form not in {"10-K", "10-K/A"}:
            continue

        fy = None
        if fiscal_years and i < len(fiscal_years) and fiscal_years[i]:
            fy = int(fiscal_years[i])
        elif report_dates and i < len(report_dates) and report_dates[i]:
            fy = int(str(report_dates[i])[:4])

        candidates.append({
            "fiscal_year": fy if fy is not None else 0,
            "filing_date": filing_dates[i] if i < len(filing_dates) else "",
            "accession_number": accession_numbers[i] if i < len(accession_numbers) else "",
            "cik": cik,
            "headers": downloader.headers,
        })

    candidates = [c for c in candidates if c["accession_number"]]

    unique_by_year = {}
    for c in sorted(candidates, key=lambda x: x["filing_date"], reverse=True):
        fy = c["fiscal_year"]
        if fy not in unique_by_year:
            unique_by_year[fy] = c

    latest = sorted(unique_by_year.values(), key=lambda x: x["fiscal_year"], reverse=True)[:2]
    return latest


def _chunk_text(text: str, chunk_size: int = 1400, overlap: int = 250) -> list[str]:
    text = (text or "").strip()
    if not text:
        return []

    chunks = []
    start = 0
    n = len(text)
    step = max(1, chunk_size - overlap)
    while start < n:
        end = min(n, start + chunk_size)
        chunks.append(text[start:end])
        if end >= n:
            break
        start += step
    return chunks


def build_10k_rag_context(company_name: str, question: str, top_k: int = 4) -> tuple[str, list[str]]:
    filings = _find_latest_two_10k_filings_api(company_name)
    if not filings:
        return "", []

    q_tokens = _tokenize_for_rag(question)
    scored_chunks = []
    used_sources = []

    for filing in filings:
        try:
            raw = _fetch_complete_submission_text(
                cik=filing["cik"],
                accession_number=filing["accession_number"],
                headers=filing["headers"],
            )
            txt = _extract_10k_text_from_submission_raw(raw)
        except Exception:
            continue

        if not txt:
            continue

        src = f"FY{filing['fiscal_year']} ({filing['filing_date']})"
        used_sources.append(src)

        for ch in _chunk_text(txt):
            c_tokens = _tokenize_for_rag(ch)
            score = len(q_tokens.intersection(c_tokens)) if q_tokens else 0
            scored_chunks.append((score, src, ch))

    if not scored_chunks:
        return "", used_sources

    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    selected = scored_chunks[:top_k]

    parts = []
    for i, (_, src, ch) in enumerate(selected, start=1):
        snippet = ch[:1200].replace("\n", " ").strip()
        parts.append(f"[{i}] Quelle={src} | Auszug: {snippet}")

    return "\n\n".join(parts), used_sources

## Query an LLM mit RAG und Übergabe konkreter Daten

In [30]:
company_name = "Meta"
# Query des Nutzers
question = "Wie haben sich aus welchen Gründen die CAPEX entwickelt?"

# Abfrage der Rohdaten und Berechnen der Bewertungsmultuples
df = load_three_metrics(BIG_TECH[company_name], 2020, 2024)
metrics_ctx = df.pivot_table(index="year", columns="metric", values="value", aggfunc="first").reset_index().to_string(index=False)

# Abfrage der Unternehmensreports und Berechnen der RAG Scores
rag_ctx, sources = build_10k_rag_context(company_name=company_name, question=question)

# Aufbauen des zusätzlichen Kontextes (neben der Query) für die LLM-Abfrage
combined_ctx = f"Unternehmen: {company_name}\n\nMetriken:\n{metrics_ctx}\n\nRAG Kontext:\n{rag_ctx}"

# Anfrage an das LLM mit dem gesamten (kombinierten) Kontext
answer = ask_llm(question, context=combined_ctx)
print(answer)

Lade Submissions für CIK 0001326801...
Lade 2 zusätzliche Filing-Archive...
Die CAPEX von Meta sind von 2020 bis 2024 stark gestiegen, mit einem deutlichen Sprung im Jahr 2022. Dies ist wahrscheinlich auf die massiven Investitionen in das Metaverse und die Infrastruktur zurückzuführen.


## Mini Dash Ansicht

In [ ]:
app = Dash(__name__)

app.layout = html.Div([
    html.H3("SEC Teaching Mini App"),
    html.Div([
        dcc.Dropdown(
            id="company_dd",
            options=[{"label": name, "value": name} for name in BIG_TECH.keys()],
            value= list(BIG_TECH.keys())[0],
            clearable=False,
            style={"width": "260px"},
        ),
        dcc.Dropdown(
            id="metric_dd",
            options=[
                {"label": "Diluted Earnings", "value": "diluted_earnings"},
                {"label": "CAPEX", "value": "capex"},
                {"label": "FCF", "value": "fcf"},
            ],
            value="fcf",
            clearable=False,
            style={"width": "220px"},
        ),
    ], style={"display": "flex", "gap": "12px", "marginBottom": "10px"}),
    dcc.Graph(id="line_fig"),
    html.Div(id="simple_label", style={"marginTop": "8px", "marginBottom": "12px"}),
    dcc.Textarea(id="chat_input", placeholder="Frage z. B.: Wie ist der FCF?", style={"width": "100%", "height": "80px"}),
    html.Button("Senden", id="send_btn", n_clicks=0, style={"marginTop": "8px"}),
    html.Div(id="chat_output", style={"marginTop": "10px", "padding": "10px", "backgroundColor": "#f6f8fa"}),
])

@app.callback(
    Output("line_fig", "figure"),
    Output("simple_label", "children"),
    Input("company_dd", "value"),
    Input("metric_dd", "value"),
)
def update_chart(company_name, metric):
    df = load_three_metrics(BIG_TECH[company_name], 2020, 2024)
    d = df[df["metric"] == metric].sort_values("year")
    fig = px.line(d, x="year", y="value", markers=True, title=f"{company_name}: {metric} pro Jahr")
    latest = d.iloc[-1]["value"] if not d.empty else None
    label = f"{company_name}: kein aktueller Wert" if latest is None or pd.isna(latest) else f"{company_name}: {metric} aktuell {'positiv' if latest > 0 else 'nicht positiv'}"
    return fig, label

@app.callback(
    Output("chat_output", "children"),
    Input("send_btn", "n_clicks"),
    State("chat_input", "value"),
    State("company_dd", "value"),
)
def update_chat(n_clicks, question, company_name):
    if not n_clicks:
        return "Noch keine Frage gestellt."

    df = load_three_metrics(BIG_TECH[company_name], 2020, 2024)
    metrics_ctx = df.pivot_table(index="year", columns="metric", values="value", aggfunc="first").reset_index().to_string(index=False)

    rag_ctx, sources = build_10k_rag_context(company_name=company_name, question=question or "")
    combined_ctx = f"Unternehmen: {company_name}\n\nMetriken:\n{metrics_ctx}"
    if rag_ctx:
        combined_ctx += f"\n\nRAG (letzte 2x 10-K):\n{rag_ctx}"

    try:
        answer = ask_llm(question or "", context=combined_ctx)
    except Exception as e:
        answer = f"LLM-Fehler: {e}"

    if sources:
        src_text = "Quellen (RAG): " + ", ".join(sources)
    else:
        src_text = "Quellen (RAG): keine 10-K Dateien gefunden"

    return [html.B("Antwort: "), answer, html.Br(), html.Small(src_text)]

In [29]:
# App im Notebook starten
app.run(jupyter_mode="inline", debug=False, port=8056)

Lade Submissions für CIK 0000320193...
Lade 1 zusätzliche Filing-Archive...
